# Logistic Regression — From Scratch, Class Imbalance & Comparison to scikit-learn

Predicts customer churn (`is_churned`) from a Spotify-style user dataset. Covers preprocessing,
a from-scratch logistic regression implementation with class-weighted loss, handling class
imbalance with SMOTE, and a final comparison against scikit-learn's `LogisticRegression`.
 

## Part A — Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

In [ ]:
data = pd.read_csv("./data/spotify_data.csv", sep=",", engine="python")
data.drop("user_id", axis=1, inplace=True)
print(data.head())

In [ ]:
X = data.drop("is_churned", axis=1)
y = data["is_churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X.columns)

In [ ]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns

print("Numeric columns:", list(num_cols))
print("Categorical columns:", list(cat_cols))

In [ ]:
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

# fit on train, transform on both (avoids leaking test-set statistics into training)
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])

X_test[num_cols] = num_imputer.transform(X_test[num_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# fit on train only, transform both — prevents data leakage
X_train_encoded = encoder.fit_transform(X_train[cat_cols])
X_test_encoded = encoder.transform(X_test[cat_cols])

X_train_encoded = pd.DataFrame(X_train_encoded, index=X_train.index)
X_test_encoded = pd.DataFrame(X_test_encoded, index=X_test.index)

X_train = X_train.drop(columns=cat_cols).join(X_train_encoded)
X_test = X_test.drop(columns=cat_cols).join(X_test_encoded)

In [ ]:
# Standardize numeric features to mean=0, std=1
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train = X_train.astype(float)
X_test = X_test.astype(float)

## Part B — Logistic Regression From Scratch (with class-weighted loss)

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
def weighted_cross_entropy_loss(y_true, y_pred, w1, w0):
    eps = 1e-10
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(w1 * y_true * np.log(y_pred) + w0 * (1 - y_true) * np.log(1 - y_pred))

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.01, epochs=1000, use_class_weights=False):
        self.lr = lr
        self.epochs = epochs
        self.use_class_weights = use_class_weights

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        n_samples, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0

        # class weights to counter label imbalance
        if self.use_class_weights:
            pos = np.sum(y == 1)
            neg = np.sum(y == 0)
            self.w1 = n_samples / (2 * pos)
            self.w0 = n_samples / (2 * neg)
        else:
            self.w1 = self.w0 = 1

        self.loss_history = []
        for _ in range(self.epochs):
            linear = np.dot(X, self.weights) + self.bias
            y_pred = sigmoid(linear)

            loss = weighted_cross_entropy_loss(y, y_pred, self.w1, self.w0)
            self.loss_history.append(loss)

            error = y_pred - y
            dw = (1 / n_samples) * np.dot(X.T, error)
            db = (1 / n_samples) * np.sum(error)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict_proba(self, X):
        linear = np.dot(X, self.weights) + self.bias
        return sigmoid(linear)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [ ]:
model = LogisticRegressionScratch(lr=0.01, epochs=2000, use_class_weights=True)
model.fit(X_train.astype(float), y_train)

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Logistic Regression from Scratch (class-weighted)")
plt.show()

## Part C — Handling Class Imbalance with SMOTE

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
smote = SMOTE(random_state=42)
X_train.columns = X_train.columns.astype(str)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

model = LogisticRegressionScratch(lr=0.01, epochs=2000, use_class_weights=True)
model.fit(X_train_res.astype(float), y_train_res)

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — SMOTE-Resampled Training Data")
plt.show()

## Part D — Comparison Against scikit-learn's LogisticRegression

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
X_train_res = pd.DataFrame(X_train_res, columns=X_train.columns.astype(str))
X_test.columns = X_test.columns.astype(str)

sk_model = LogisticRegression(max_iter=2000, class_weight="balanced")
sk_model.fit(X_train_res, y_train_res)

y_pred = sk_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — scikit-learn LogisticRegression")
plt.show()